In [2]:
from jupyter_dash import JupyterDash
import dash_leaflet as dl
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output
import plotly.express as px
import pandas as pd
import base64

JupyterDash.infer_jupyter_proxy_config()

from CSCRUD import AnimalShelter

db = AnimalShelter()

df = pd.DataFrame.from_records(db.read({}))

if "_id" in df.columns:
    df.drop(columns=["_id"], inplace=True)

print("Database:", db.database.name)
print("Collection:", db.collection.name)
print("Records loaded:", len(df))

if df.empty:
    raise RuntimeError(
        "No records were returned from aac.animals. "
        "Check MongoDB authentication and verify that the "
        "original animal records are in this collection."
    )


Database: aac
Collection: animals
Records loaded: 10000


In [4]:
import base64
import os
import pandas as pd
import plotly.express as px
import dash_leaflet as dl

from jupyter_dash import JupyterDash
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output

JupyterDash.infer_jupyter_proxy_config()

FIELD_MAP = {
    "current_age": "age_upon_outcome",
    "animal_id": "animal_id",
    "animal_type": "animal_type",
    "breed": "breed",
    "color": "color",
    "date_of_birth": "date_of_birth",
    "date_time": "datetime",
    "month_year": "monthyear",
    "name": "name",
    "outcome": "outcome_type",
    "outcome_reason": "outcome_subtype",
    "latitude": "location_lat",
    "longitude": "location_long",
    "age_at_outcome_in_weeks": "age_upon_outcome_in_weeks"
}

DISPLAY_LABELS = {
    "current_age": "Age",
    "animal_id": "Animal ID",
    "animal_type": "Animal Type",
    "breed": "Breed",
    "color": "Color",
    "date_of_birth": "Date of Birth",
    "date_time": "Date and Time",
    "month_year": "Month/Year",
    "name": "Name",
    "outcome": "Outcome",
    "outcome_reason": "Outcome Reason",
    "latitude": "Latitude",
    "longitude": "Longitude",
    "age_at_outcome_in_weeks": "Age at Outcome (Weeks)"
}

missing_columns = [
    column for column in FIELD_MAP.values()
    if column not in df.columns
]

if missing_columns:
    raise KeyError(
        f"Missing columns in MongoDB records: {missing_columns}"
    )

dashboard_df = pd.DataFrame(index=df.index)

for display_name, database_name in FIELD_MAP.items():
    dashboard_df[display_name] = df[database_name]

NUMERIC_DECIMALS = {
    "latitude": 4,
    "longitude": 4,
    "age_at_outcome_in_weeks": 1
}

for column in NUMERIC_DECIMALS:
    dashboard_df[column] = pd.to_numeric(
        dashboard_df[column], errors="coerce"
    )

for column in dashboard_df.columns:
    if column not in NUMERIC_DECIMALS:
        dashboard_df[column] = (
            dashboard_df[column]
            .fillna("")
            .astype(str)
            .str.strip()
        )

dashboard_df["animal_type"] = (
    dashboard_df["animal_type"].str.title()
)

dashboard_df = dashboard_df.reset_index(drop=True)
dashboard_df["internal_id"] = dashboard_df.index.astype(str)

DISPLAY_FIELDS = list(FIELD_MAP.keys())

COLUMN_WIDTHS = {
    "current_age": 125,
    "animal_id": 125,
    "animal_type": 125,
    "breed": 230,
    "color": 150,
    "date_of_birth": 140,
    "date_time": 170,
    "month_year": 140,
    "name": 130,
    "outcome": 150,
    "outcome_reason": 160,
    "latitude": 120,
    "longitude": 120,
    "age_at_outcome_in_weeks": 160
}

TABLE_WIDTH = sum(COLUMN_WIDTHS.values())
PIE_TOP_BREEDS = 8
OTHER_LABEL = "Other breeds"

filter_df = dashboard_df.copy()

for column, decimals in NUMERIC_DECIMALS.items():
    filter_df[column] = filter_df[column].round(decimals)

def dropdown_options(column_name):
    values = [
        str(value)
        for value in filter_df[column_name].unique()
        if str(value).strip() not in ("", "nan", "None")
    ]

    if column_name in NUMERIC_DECIMALS:
        return sorted(values, key=float)

    return sorted(values, key=str.casefold)

def make_filter_dropdown(column_name):
    return html.Div(
        style={
            "width": f"{COLUMN_WIDTHS[column_name]}px",
            "minWidth": f"{COLUMN_WIDTHS[column_name]}px",
            "flex": "none",
            "padding": "0 3px",
            "boxSizing": "border-box"
        },
        children=[
            dcc.Dropdown(
                id=f"filter-{column_name}",
                options=[
                    {"label": value, "value": value}
                    for value in dropdown_options(column_name)
                ],
                placeholder=f"Filter {DISPLAY_LABELS[column_name]}",
                multi=True,
                searchable=True,
                clearable=True,
                style={"fontSize": "12px"}
            )
        ]
    )

def table_column(column_name):
    column = {
        "name": DISPLAY_LABELS[column_name],
        "id": column_name
    }

    if column_name in NUMERIC_DECIMALS:
        column["type"] = "numeric"
        column["format"] = {
            "specifier": f".{NUMERIC_DECIMALS[column_name]}f"
        }

    return column

logo_path = "Grazioso Salvare Logo.png"

try:
    with open(logo_path, "rb") as image_file:
        encoded_image = base64.b64encode(
            image_file.read()
        ).decode("utf-8")
except FileNotFoundError:
    encoded_image = None
    print(f"Logo not found at {os.path.abspath(logo_path)}")

print("Dashboard records:", len(dashboard_df))
print("Displayed columns:", len(DISPLAY_FIELDS))


Dashboard records: 10000
Displayed columns: 14


In [5]:
app = JupyterDash(__name__)

table_records = filter_df[DISPLAY_FIELDS].to_dict("records")

pie_type_options = [
    {"label": value, "value": value}
    for value in dropdown_options("animal_type")
]

pie_breed_options = [
    {"label": value, "value": value}
    for value in dropdown_options("breed")
]

app.layout = html.Div(
    style={
        "backgroundColor": "#f3f5f8",
        "minHeight": "100vh",
        "padding": "20px",
        "fontFamily": "Arial, sans-serif",
        "color": "#243447"
    },
    children=[
        html.H1(
            "CS-340 Animal Shelter Dashboard",
            style={"textAlign": "center"}
        ),

        html.P(
            "Animal records, breed distribution, and locations",
            style={
                "textAlign": "center",
                "color": "#667085"
            }
        ),

        html.Hr(),

        html.Div(
            style={"textAlign": "center", "padding": "10px"},
            children=[
                html.Img(
                    src="data:image/png;base64," + encoded_image,
                    style={
                        "maxWidth": "280px",
                        "maxHeight": "140px",
                        "height": "auto"
                    }
                ) if encoded_image else html.Div()
            ]
        ),

        html.Div(
            style={
                "display": "flex",
                "gap": "20px",
                "flexWrap": "wrap",
                "alignItems": "stretch"
            },
            children=[
                html.Div(
                    style={
                        "flex": "1",
                        "minWidth": "320px",
                        "backgroundColor": "white",
                        "padding": "15px",
                        "borderRadius": "10px",
                        "boxShadow": "0 2px 6px #00000010"
                    },
                    children=[
                        html.H2("Animal Locations"),
                        html.P(
                            "Map markers represent records in the table "
                            "after table filters and sorting are applied."
                        ),
                        dl.Map(
                            id="animal-map",
                            center=[30.75, -97.48],
                            zoom=10,
                            style={
                                "width": "100%",
                                "height": "460px"
                            },
                            children=[dl.TileLayer()]
                        )
                    ]
                ),

                html.Div(
                    style={
                        "flex": "1",
                        "minWidth": "320px",
                        "backgroundColor": "white",
                        "padding": "15px",
                        "borderRadius": "10px",
                        "boxShadow": "0 2px 6px #00000010"
                    },
                    children=[
                        html.H2("Animal Breed Distribution"),

                        html.Label("Filter by Animal Type"),
                        dcc.Dropdown(
                            id="pie-animal-type",
                            options=pie_type_options,
                            placeholder="All animal types",
                            clearable=True,
                            multi=True
                        ),

                        html.Br(),

                        html.Label("Filter by Breed"),
                        dcc.Dropdown(
                            id="pie-breed",
                            options=pie_breed_options,
                            placeholder="All breeds",
                            clearable=True,
                            multi=True
                        ),

                        dcc.Graph(
                            id="pie-chart",
                            style={"height": "430px"}
                        )
                    ]
                )
            ]
        ),

        html.Br(),

        html.Div(
            style={
                "backgroundColor": "white",
                "padding": "15px",
                "borderRadius": "10px",
                "boxShadow": "0 2px 6px #00000010"
            },
            children=[
                html.H2("Animal Records"),

                html.P(
                    "Select one or more values in any column. "
                    "Filters from different columns apply together. "
                    "The chart filters above do not affect this table."
                ),

                html.Button(
                    "Reset Table Filters",
                    id="reset-filters",
                    n_clicks=0,
                    style={
                        "backgroundColor": "#34495e",
                        "color": "white",
                        "padding": "10px 16px",
                        "border": "none",
                        "borderRadius": "5px",
                        "cursor": "pointer"
                    }
                ),

                html.Div(
                    style={
                        "overflowX": "auto",
                        "marginTop": "15px"
                    },
                    children=[
                        html.Div(
                            style={"width": f"{TABLE_WIDTH}px"},
                            children=[
                                html.Div(
                                    style={
                                        "display": "flex",
                                        "paddingBottom": "8px"
                                    },
                                    children=[
                                        make_filter_dropdown(field)
                                        for field in DISPLAY_FIELDS
                                    ]
                                ),

                                dash_table.DataTable(
                                    id="datatable-id",
                                    columns=[
                                        table_column(field)
                                        for field in DISPLAY_FIELDS
                                    ],
                                    data=table_records,

                                    row_selectable=False,
                                    cell_selectable=False,

                                    sort_action="native",
                                    sort_mode="multi",
                                    page_action="native",
                                    page_size=10,

                                    style_table={
                                        "width": f"{TABLE_WIDTH}px",
                                        "overflowX": "auto"
                                    },
                                    style_header={
                                        "backgroundColor": "#34495e",
                                        "color": "white",
                                        "fontWeight": "bold",
                                        "textAlign": "left",
                                        "padding": "10px"
                                    },
                                    style_cell={
                                        "padding": "8px",
                                        "whiteSpace": "normal",
                                        "height": "auto",
                                        "textAlign": "left",
                                        "minWidth": "100px"
                                    },
                                    style_cell_conditional=[
                                        {
                                            "if": {
                                                "column_id": field
                                            },
                                            "width": (
                                                f"{COLUMN_WIDTHS[field]}px"
                                            ),
                                            "minWidth": (
                                                f"{COLUMN_WIDTHS[field]}px"
                                            ),
                                            "maxWidth": (
                                                f"{COLUMN_WIDTHS[field]}px"
                                            )
                                        }
                                        for field in DISPLAY_FIELDS
                                    ]
                                )
                            ]
                        )
                    ]
                )
            ]
        )
    ]
)

print("Dashboard layout created.")


Dashboard layout created.


In [6]:
@app.callback(
    Output("datatable-id", "data"),
    [
        Input(f"filter-{field}", "value")
        for field in DISPLAY_FIELDS
    ]
)
def update_table(*selected_values):
    filtered = filter_df[DISPLAY_FIELDS].copy()

    for field, values in zip(DISPLAY_FIELDS, selected_values):
        if values:
            selected_strings = [str(value) for value in values]

            if field in NUMERIC_DECIMALS:
                numeric_values = pd.to_numeric(
                    filtered[field], errors="coerce"
                ).round(NUMERIC_DECIMALS[field])

                filtered = filtered[
                    numeric_values.astype(str).isin(selected_strings)
                ]
            else:
                filtered = filtered[
                    filtered[field].astype(str).isin(selected_strings)
                ]

    return filtered.to_dict("records")


@app.callback(
    [
        Output(f"filter-{field}", "value")
        for field in DISPLAY_FIELDS
    ],
    Input("reset-filters", "n_clicks"),
    prevent_initial_call=True
)
def reset_table_filters(n_clicks):
    return [None] * len(DISPLAY_FIELDS)

print("Table filter callbacks registered.")


Table filter callbacks registered.


In [7]:
@app.callback(
    Output("pie-chart", "figure"),
    [
        Input("pie-animal-type", "value"),
        Input("pie-breed", "value")
    ]
)
def update_pie_chart(animal_types, breeds):
    filtered = dashboard_df.copy()

    if animal_types:
        filtered = filtered[
            filtered["animal_type"].isin(animal_types)
        ]

    if breeds:
        filtered = filtered[
            filtered["breed"].isin(breeds)
        ]

    counts = (
        filtered.loc[filtered["breed"] != "", "breed"]
        .value_counts()
    )

    top = counts.head(PIE_TOP_BREEDS)
    other_total = int(counts.iloc[PIE_TOP_BREEDS:].sum())

    labels = top.index.tolist()
    values = top.astype(int).tolist()

    if other_total > 0:
        labels.append(OTHER_LABEL)
        values.append(other_total)

    if not values:
        fig = px.pie()
        fig.update_layout(
            title="No animals match these chart filters",
            template="plotly_white",
            height=430
        )
        return fig

    if animal_types and breeds:
        title = "Breed Distribution for Selected Types and Breeds"
    elif animal_types:
        title = "Breed Distribution by Selected Animal Types"
    elif breeds:
        title = "Distribution of Selected Breeds"
    else:
        title = "Animal Breed Distribution"

    fig = px.pie(
        names=labels,
        values=values,
        title=title,
        hole=0.30
    )

    fig.update_traces(
        sort=False,
        textposition="inside",
        textinfo="label+percent",
        hovertemplate=(
            "<b>%{label}</b><br>"
            "Animals: %{value}<br>"
            "Percentage: %{percent}"
            "<extra></extra>"
        )
    )

    fig.update_layout(
        template="plotly_white",
        height=430,
        title={"x": 0.5},
        legend_title_text="Breed"
    )

    return fig

print("Pie-chart callbacks registered.")


Pie-chart callbacks registered.


In [8]:
@app.callback(
    Output("animal-map", "children"),
    Input("datatable-id", "derived_virtual_data")
)
def update_map(table_data):
    children = [dl.TileLayer()]

    if not table_data:
        return children

    map_df = pd.DataFrame(table_data)

    if map_df.empty:
        return children

    for _, row in map_df.iterrows():
        latitude = pd.to_numeric(
            row.get("latitude"), errors="coerce"
        )
        longitude = pd.to_numeric(
            row.get("longitude"), errors="coerce"
        )

        if pd.isna(latitude) or pd.isna(longitude):
            continue

        if not (
            -90 <= latitude <= 90
            and -180 <= longitude <= 180
        ):
            continue

        name = str(row.get("name", "") or "Unknown")
        breed = str(row.get("breed", "") or "Unknown breed")
        animal_type = str(row.get("animal_type", "") or "Unknown")
        animal_id = str(row.get("animal_id", "") or "Unknown")

        children.append(
            dl.Marker(
                position=[float(latitude), float(longitude)],
                children=[
                    dl.Tooltip(f"{name} — {breed}"),
                    dl.Popup(
                        html.Div([
                            html.H4("Animal Information"),
                            html.P(f"Name: {name}"),
                            html.P(f"Animal ID: {animal_id}"),
                            html.P(f"Animal Type: {animal_type}"),
                            html.P(f"Breed: {breed}")
                        ])
                    )
                ]
            )
        )

    return children

print("Map callback registered.")


Map callback registered.


In [10]:
import dash
import jupyter_dash

print("Dash version:", dash.__version__)
print("JupyterDash imported successfully")

Dash version: 2.8.1
JupyterDash imported successfully


In [11]:
app.run_server(mode="inline", port=8055, debug=False)